# AUP-ZU3 — LeNet Inference Through DDR (PYNQ)

This notebook runs the **custom LeNet HLS IP** built from `spec/hls_spec.md` and `spec/hardware_spec.md`.

**Data path:**

```
CPU  ──write 784 × float32──▶  DDR (source buffer)
                                  │  DMA MM2S reads
                                  ▼
                            lenet_axis  (HLS IP, AXI-Stream 32-bit)
                                  │  DMA S2MM writes 10 × float32
                                  ▼
                              DDR (destination buffer)  ──read──▶  CPU compares with Keras
```

One DMA transfer = one image. The IP is free-running (`ap_ctrl_none`): no registers to program, it starts as soon as pixels arrive and asserts `TLAST` on the 10th output word.

**Design facts (from the specs):**

| Item | Value |
|------|-------|
| Board | AUP-ZU3 8GB (`xczu3eg-sfvc784-2-e`) |
| AXI DMA control base | `0xA000_0000` (64 KB) |
| Input per image | 784 × float32, pixel / 255.0, row-major |
| Output per image | 10 × float32 softmax probabilities |
| Golden reference | `keras_probs` in `mnist_test_1000.npz` (Keras top-1 = 96.70 %) |
| PL clock | 100 MHz |

---
## 1. Prerequisites

Run this notebook **on the AUP-ZU3 board** in a PYNQ image (Python 3, `pynq` installed).

Copy these three files into the **same directory as this notebook**:

- `lenet.bit` — the bitstream
- `lenet.hwh` — the hardware hand-off metadata (same base name as the `.bit`)
- `mnist_test_1000.npz` — test images, labels and Keras outputs (from `data/`)

```bash
cp vivado/aup_zu3_lenet.runs/impl_1/design_1_wrapper.bit                     pynq/lenet.bit
cp vivado/aup_zu3_lenet.gen/sources_1/bd/design_1/hw_handoff/design_1.hwh    pynq/lenet.hwh
cp data/mnist_test_1000.npz                                                   pynq/
```

---
## 2. Load the overlay

`Overlay(...)` downloads the bitstream to the PL and parses the `.hwh`. `lenet_axis_0` has no AXI-Lite port, so only `axi_dma_0` appears as a driver.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time

overlay = Overlay("lenet.bit")
print(list(overlay.ip_dict.keys()))

## 3. Grab a handle to the AXI DMA

- `sendchannel` → **MM2S** (DDR → stream, feeds `lenet_axis`)
- `recvchannel` → **S2MM** (stream → DDR, drains `lenet_axis`)

In [ ]:
dma = overlay.axi_dma_0
mm2s = dma.sendchannel
s2mm = dma.recvchannel
print("AXI DMA ready:", dma)

## 4. Load the test set

Pixels are stored as `uint8`; the model input is `pixel / 255.0` as float32 — the same scaling used in training and in the HLS testbench.

In [ ]:
data = np.load("mnist_test_1000.npz")
x = data["x"].reshape(-1, 784).astype(np.float32) / 255.0
y = data["y"]
keras_probs = data["keras_probs"]
N = len(y)

keras_acc = float(np.mean(keras_probs.argmax(axis=1) == y))
print(f"{N} images, Keras top-1 accuracy = {keras_acc:.4f}")

## 5. Allocate DDR buffers

`allocate()` returns physically contiguous DDR the DMA can reach. `float32` matches the 32-bit stream word, so no bit packing is needed.

In [ ]:
in_buf = allocate(shape=(784,), dtype=np.float32)
out_buf = allocate(shape=(10,), dtype=np.float32)

## 6. One inference

Same order as Lab2: arm the receive (S2MM) side before starting the send (MM2S) side. `out_buf` is cleared first so a failed transfer is obvious. `flush()` / `invalidate()` keep the CPU cache and DDR consistent.

In [ ]:
def infer(img):
    in_buf[:] = img
    out_buf[:] = 0
    in_buf.flush()
    out_buf.flush()
    s2mm.transfer(out_buf)   # arm receive first
    mm2s.transfer(in_buf)    # then send the image
    mm2s.wait()
    s2mm.wait()
    out_buf.invalidate()
    return np.array(out_buf)


i = 0
p = infer(x[i])
for row in x[i].reshape(28, 28)[::2]:
    print("".join("#" if v > 0.5 else "." for v in row[::2]))
print(f"\nlabel = {y[i]}   board = {p.argmax()}   keras = {keras_probs[i].argmax()}")
print("class   board     keras")
for c in range(10):
    print(f"  {c}    {p[c]:.4f}    {keras_probs[i][c]:.4f}")

## 7. Run all images

Each image is one DMA round trip. The measured time includes Python and DMA overhead, so it is an upper bound on the IP latency (see `hls/REPORT.md` for the cycle count).

In [ ]:
board_probs = np.zeros((N, 10), dtype=np.float32)
t0 = time.perf_counter()
for i in range(N):
    board_probs[i] = infer(x[i])
t1 = time.perf_counter()
print(f"{N} images in {t1 - t0:.2f} s  ({(t1 - t0) / N * 1e6:.1f} us / image incl. Python + DMA)")

## 8. Verify against Keras

Pass criteria:

- top-1 agreement with Keras ≥ 99 %
- board accuracy ≥ Keras accuracy − 0.5 %

In [ ]:
pred = board_probs.argmax(axis=1)
keras_pred = keras_probs.argmax(axis=1)
board_acc = float(np.mean(pred == y))
agree = float(np.mean(pred == keras_pred))
max_err = float(np.abs(board_probs - keras_probs).max())
sum_err = float(np.abs(board_probs.sum(axis=1) - 1.0).max())

print(f"board accuracy   = {board_acc:.4f}   (Keras {keras_acc:.4f})")
print(f"agreement        = {agree:.4f}")
print(f"max |p - p_keras| = {max_err:.4f}")
print(f"max |sum(p) - 1|  = {sum_err:.4f}")

bad = np.flatnonzero(pred != keras_pred)
for i in bad[:5]:
    print(f"  image {i}: label {y[i]}, board {pred[i]}, keras {keras_pred[i]}")

if agree >= 0.99 and board_acc >= keras_acc - 0.005:
    print(f"\n✅ PASS — LeNet on the FPGA matches Keras on {N - len(bad)}/{N} images.")
else:
    print(f"\n❌ FAIL — {len(bad)} images disagree with Keras.")

## 9. Clean up

Free the contiguous DDR buffers so the memory returns to the allocator.

In [ ]:
in_buf.freebuffer()
out_buf.freebuffer()
print("Buffers freed.")

## Troubleshooting

| Symptom | Likely cause / fix |
|---------|--------------------|
| `s2mm.wait()` hangs forever | The IP never asserted `TLAST`, or it is waiting for more input. Re-run the overlay cell to reset the PL, then retry. |
| `mm2s.wait()` hangs forever | `lenet_axis` is not accepting data (`TREADY` low): reset not released, or the IP is still stalled on its output. Re-load the overlay. |
| DMA error / `RuntimeError: DMA channel not idle` | The IP sent more than 10 words before `TLAST` (S2MM buffer overflow), or a previous transfer did not finish. Re-load the overlay. |
| `KeyError: 'axi_dma_0'` | The `.hwh` doesn't match the loaded `.bit`, or the DMA instance has a different name. Check `overlay.ip_dict.keys()`. |
| Output all zeros or wrong but csim/cosim passed | Check the `.bit` / `.hwh` are from the same build. A previous attempt on this board saw wrong results caused by registers relying on power-on values (`config_rtl -reset all` in HLS fixes it). |
| Accuracy far below Keras, outputs look shuffled | Pixel order or scaling mismatch: input must be row-major `pixel / 255.0`. |